# 03.5 预训练文本模型入门（可选）（Pretrained Text Models Optional）

这一节是扩展内容

真实 NLP 工作里，很多任务并不是从零开始训练文本模型，而是直接复用一个 `pretrained model  

但当前仓库的离线环境里没有 `transformers` 包，也不能临时下载模型。  

所以本 notebook 会做两件事

1. 解释真实世界里的预训练文本分类流程
2. 用一个离线可运行的 `backbone + head` 模拟练习关键概念

## 学习目标

学完后你应该能

1. classifier head` 的角色
2. 理解 `freeze backbone
3. 看懂一个标准文本分类工作流
4. 在离线环境下跑通一个结构上相似的简化版本

In [ ]:
import importlib.util
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
HAS_TRANSFORMERS = importlib.util.find_spec("transformers") is not None
print("transformers available / 是否安装 transformers:", HAS_TRANSFORMERS)

## 1. 真实世界工作流长什么样

如果环境里已经有 `transformers` 和本地缓存模型，一个典型流程通常是：  

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

batch = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
logits = model(**batch).logits
```

你现在最需要先掌握的是结构，而不是某个具体库的调用细节。  


## 2. 准备一个小型文本分类数据集

sentiment classification 数据集做结构演示。  

标签约定

- `1` 表示正向（positive）
- `0` 表示负向（negative）

In [ ]:
positive_texts = [
    "good movie",
    "great film",
    "excellent story",
    "amazing plot",
    "nice acting",
    "fun show",
    "enjoyable movie",
    "wonderful film",
    "really good story",
    "very nice plot",
    "love this movie",
    "great acting",
]

negative_texts = [
    "bad movie",
    "terrible film",
    "awful story",
    "boring plot",
    "poor acting",
    "dull show",
    "unpleasant movie",
    "horrible film",
    "really bad story",
    "very poor plot",
    "hate this movie",
    "terrible acting",
]

texts = positive_texts + negative_texts
labels = [1] * len(positive_texts) + [0] * len(negative_texts)

print("dataset size =", len(texts))
print("positive ratio / 正样本比例 =", sum(labels) / len(labels))
print("example text =", texts[0], "| label =", labels[0])

## 分词器 的最小模拟

真实预训练模型通常会自带 tokenizer。  

这里我们用最简单的空格分词做概念演示

In [ ]:
def tokenize(text):
    return text.lower().split()


special_tokens = ["<pad>", "<unk>"]
vocab = sorted({token for text in texts for token in tokenize(text)})
vocab = special_tokens + vocab
stoi = {token: idx for idx, token in enumerate(vocab)}

print("vocab size =", len(vocab))
print("first 12 vocab items =", vocab[:12])

In [ ]:
def encode(text, max_len=4):
    tokens = tokenize(text)
    ids = [stoi.get(token, stoi["<unk>"]) for token in tokens][:max_len]
    attention_mask = [1] * len(ids)

    while len(ids) < max_len:
        ids.append(stoi["<pad>"])
        attention_mask.append(0)

    return ids, attention_mask


encoded = [encode(text) for text in texts]
input_ids = torch.tensor([item[0] for item in encoded], dtype=torch.long)
attention_mask = torch.tensor([item[1] for item in encoded], dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.long)

print("input_ids.shape =", input_ids.shape)
print("attention_mask.shape =", attention_mask.shape)
print("example input_ids =", input_ids[0])
print("example attention_mask =", attention_mask[0])

注意力掩码` 在真实模型里很常见。  

它告诉模型哪些位置是真实 token，哪些位置只是 padding。  


## 4. 构造一个“模拟预训练” backbone

这里我们不会假装在做真实 BERT 预训练。  

already useful representations”，然后冻结它们。  

这能帮助你理解 `frozen backbone + classifier head` 的工作方式。  


In [ ]:
embedding_dim = 4
pretrained_weights = torch.zeros(len(vocab), embedding_dim)

positive_words = {"good", "great", "excellent", "amazing", "nice", "fun", "enjoyable", "wonderful", "love"}
negative_words = {"bad", "terrible", "awful", "boring", "poor", "dull", "unpleasant", "horrible", "hate"}
intensifiers = {"really", "very"}
nouns = {"movie", "film", "story", "plot", "acting", "show", "this"}

for token, idx in stoi.items():
    if token in positive_words:
        pretrained_weights[idx] = torch.tensor([1.6, 0.1, 0.4, 0.0])
    elif token in negative_words:
        pretrained_weights[idx] = torch.tensor([0.1, 1.6, 0.4, 0.0])
    elif token in intensifiers:
        pretrained_weights[idx] = torch.tensor([0.3, 0.3, 1.2, 0.0])
    elif token in nouns:
        pretrained_weights[idx] = torch.tensor([0.0, 0.0, 0.1, 0.3])
    else:
        pretrained_weights[idx] = torch.tensor([0.0, 0.0, 0.0, 0.0])

print("pretrained_weights.shape =", pretrained_weights.shape)
print("embedding for 'good' =", pretrained_weights[stoi['good']])
print("embedding for 'bad' =", pretrained_weights[stoi['bad']])

In [ ]:
class FrozenTextBackbone(nn.Module):
    def __init__(self, embedding_weights, freeze=True):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights,
            freeze=freeze,
            padding_idx=stoi["<pad>"],
        )

    def forward(self, input_ids, attention_mask):
        token_embeddings = self.embedding(input_ids)
        mask = attention_mask.unsqueeze(-1)
        pooled = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return pooled


class TextClassifier(nn.Module):
    def __init__(self, embedding_weights, freeze_backbone=True):
        super().__init__()
        self.backbone = FrozenTextBackbone(embedding_weights, freeze=freeze_backbone)
        self.head = nn.Linear(embedding_weights.size(1), 2)

    def forward(self, input_ids, attention_mask):
        pooled = self.backbone(input_ids, attention_mask)
        return self.head(pooled)


probe_model = TextClassifier(pretrained_weights, freeze_backbone=True)
probe_logits = probe_model(input_ids[:3], attention_mask[:3])
print("probe_logits.shape =", probe_logits.shape)

## 只训练分类头

先冻结 backbone，只训练最后的线性层。  

这和很多真实预训练模型的第一步非常像。  


In [ ]:
X_train_ids, X_val_ids, X_train_mask, X_val_mask, y_train, y_val = train_test_split(
    input_ids,
    attention_mask,
    labels_tensor,
    test_size=0.25,
    random_state=42,
    stratify=labels_tensor,
)

train_loader = DataLoader(
    TensorDataset(X_train_ids, X_train_mask, y_train),
    batch_size=8,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(X_val_ids, X_val_mask, y_val),
    batch_size=8,
    shuffle=False,
)

print("train size =", len(X_train_ids))
print("val size =", len(X_val_ids))

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for batch_ids, batch_mask, batch_labels in loader:
        with torch.set_grad_enabled(is_train):
            logits = model(batch_ids, batch_mask)
            loss = loss_fn(logits, batch_labels)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * batch_ids.size(0)
        total_correct += (preds == batch_labels).sum().item()
        total_items += batch_ids.size(0)

    return total_loss / total_items, total_correct / total_items


head_only_model = TextClassifier(pretrained_weights, freeze_backbone=True)
trainable_params = sum(p.numel() for p in head_only_model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in head_only_model.parameters())
print("trainable params =", trainable_params)
print("all params =", all_params)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(head_only_model.parameters(), lr=0.05)

for epoch in range(1, 11):
    train_loss, train_acc = run_epoch(head_only_model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(head_only_model, val_loader, loss_fn, optimizer=None)
    print(
        f"head-only epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

## 微调

微调` 的意思是：  

- 不只训练 head
- 连 backbone 的参数也一起更新

In [ ]:
finetune_model = TextClassifier(pretrained_weights.clone(), freeze_backbone=False)
trainable_params = sum(p.numel() for p in finetune_model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in finetune_model.parameters())
print("trainable params after unfreezing =", trainable_params)
print("all params =", all_params)

optimizer = torch.optim.Adam(finetune_model.parameters(), lr=0.02)
for epoch in range(1, 7):
    train_loss, train_acc = run_epoch(finetune_model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(finetune_model, val_loader, loss_fn, optimizer=None)
    print(
        f"fine-tune epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

In [ ]:
sample_texts = ["great movie", "awful film", "really good story", "very poor plot"]
sample_encoded = [encode(text) for text in sample_texts]
sample_ids = torch.tensor([item[0] for item in sample_encoded], dtype=torch.long)
sample_mask = torch.tensor([item[1] for item in sample_encoded], dtype=torch.float32)

head_only_preds = head_only_model(sample_ids, sample_mask).argmax(dim=1)
finetune_preds = finetune_model(sample_ids, sample_mask).argmax(dim=1)

print("sample_texts =", sample_texts)
print("head_only_preds =", head_only_preds)
print("finetune_preds =", finetune_preds)

In [ ]:
# 练习 1
# 如果 batch 经过 tokenizer 之后得到 input_ids.shape == (8, 12)，
# If a batch after tokenization has input_ids.shape == (8, 12),
# 那么 attention_mask.shape 通常是多少？
# what is the usual shape of attention_mask?

练习 1 参考答案

- `attention_mask.shape == (8, 12)`

因为 attention mask 要逐位置告诉模型哪些 token 有效。  


In [ ]:
# 练习 2
# `freeze backbone` 和 `fine-tune` 的核心区别是什么？
# What is the core difference between `freeze backbone` and `fine-tune`?

练习 2 参考答案

- `freeze backbone`：只更新最后任务头
- `fine-tune`：连 backbone 参数也一起更新

通常 `freeze backbone` 更稳、更省算力；`fine-tune` 更灵活，但更容易过拟合，也更依赖学习率设置。  


## 7. 小结

这一节最重要的不是记住某个具体库的函数名，而是抓住结构：  

1. `tokenizer` 把文本变成 token ids
2. `attention mask` 标记真实 token 和 padding
3. `backbone` 负责提取文本表示
4. `classifier head` 负责把表示映射到任务标签
5. `freeze backbone` 和 `fine-tune` 是两种常见训练策略

如果你以后切到真正的 `transformers` 环境，这份 notebook 的结构会直接迁移过去。  
